### Problem Statement 4D

4D. Building a Recommendation System with CBOW Word Embeddings
Problem: Use CBOW word embeddings to build a content-based recommendation system. For example, recommend similar articles or tweets based on the embeddings of keywords in a document.
Steps:
1. Data Collection and Preprocessing: Gather a dataset with textual content. Tokenize, clean, prepare for training.
2. Train CBOW Model:
a. Train the Word2Vec model using CBOW (sg=0) to learn word embeddings.
b. For a given article (or tweet), calculate the average of the embeddings of all the words in the document.
c. For each document in the dataset, calculate the cosine similarity between the document's average word embedding and the user's input embedding.
d. Document Representation: Represent each document as a vector by averaging the embeddings of all words in the document.
3. Recommendation Logic: Recommend the most similar documents using cosine similarity between their vector representations.

In [ ]:
# Step 1: Import libraries
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Step 2: Load dataset
data = pd.read_csv(
    "../LP IV Updated Datasets/CBOW/twitter_dataset.csv"
).head(1000)

print(data.head())

In [ ]:
# Step 3: Clean and tokenize tweets
documents = []

for text in data['Text']:
    words = [
        w.lower() for w in str(text).split()
        if w.isalpha()
    ]

    documents.append(words)

print("Total documents:", len(documents))
print("Sample document:", documents[0])

In [ ]:
# Step 4: Train CBOW model
cbow_model = Word2Vec(
    documents,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    epochs=10
)

print("Vocabulary size:", len(cbow_model.wv))

In [ ]:
# Step 5: Convert documents into vectors
def get_average_embedding(words):

    vectors = [
        cbow_model.wv[w]
        for w in words
        if w in cbow_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)


doc_vectors = np.array([
    get_average_embedding(doc)
    for doc in documents
])

print("Document matrix shape:", doc_vectors.shape)

In [ ]:
# Step 6: Recommendation function
def recommend(query_text, top_k=3):

    print("\nUser Query:", query_text)

    query_words = [
        w.lower() for w in query_text.split()
        if w.isalpha()
    ]

    # Convert query into vector
    query_vector = get_average_embedding(query_words).reshape(1, -1)

    # Calculate cosine similarity
    scores = cosine_similarity(
        query_vector, doc_vectors
    ).flatten()

    # Get top matching documents
    top_indices = scores.argsort()[::-1][:top_k]

    print("\nTop Recommendations:")

    for rank, idx in enumerate(top_indices, 1):
        print(rank, "Score:", scores[idx])
        print(data['Text'].iloc[idx])

In [ ]:
# Step 7: Test recommendations
recommend("happy sunny day with great friends")

recommend("new technology and software updates")